# Phase 5 : Modélisation - Création du système de recommandation

**Objectif :** Implémenter et entraîner le modèle de filtrage collaboratif.

## Algorithme : Similarité Cosinus
1. Créer la matrice utilisateur-film
2. Calculer la similarité cosinus entre tous les utilisateurs
3. Pour un utilisateur donné : trouver ses voisins les plus proches
4. Prédire les notes par moyenne pondérée
5. Recommander les meilleurs films non notés

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ajouter le backend au path pour importer la classe
sys.path.append(os.path.abspath('..'))
from backend.recommender import MovieRecommender

# Chargement des données
movies = pd.read_csv('../data/movies_clean.csv')
ratings = pd.read_csv('../data/ratings_clean.csv')

print('✅ Bibliothèques et données chargées')

In [ ]:
# ENTRAÎNEMENT DU MODÈLE
print('='*60)
print('ENTRAÎNEMENT DU SYSTÈME DE RECOMMANDATION')
print('='*60)

recommender = MovieRecommender()
recommender.fit(ratings, movies)

In [ ]:
# TEST AVEC L'UTILISATEUR 1
user_id = 1

print(f'🎯 Recommandations pour l\'utilisateur {user_id}')
print('='*60)

# Films déjà notés
print(f'\n📌 Films notés par l\'utilisateur {user_id} :')
user_movies = recommender.get_user_ratings(user_id)
for movie in user_movies[:5]:
    stars = '⭐' * int(movie['rating'])
    print(f'  {movie["title"]} {stars} ({movie["rating"]}/5)')

# Recommandations
print(f'\n🎬 Recommandations :')
recommendations = recommender.recommend(user_id, n_recommendations=5)
for i, movie in enumerate(recommendations, 1):
    print(f'  {i}. {movie["title"]}')
    print(f'     Genre : {movie["genres"]}')
    print(f'     Note prédite : {movie["predicted_rating"]}/5\n')

In [ ]:
# TEST AVEC PLUSIEURS UTILISATEURS
test_users = [1, 50, 100, 200]

for uid in test_users:
    print(f'\n{"="*50}')
    print(f'🆔 Utilisateur {uid}')
    print(f'{"="*50}')

    # Films notés (top 3)
    user_movies = recommender.get_user_ratings(uid)
    print('\nAime :')
    for movie in user_movies[:3]:
        print(f'  ✅ {movie["title"]} ({movie["rating"]}/5)')

    # Recommandations
    recommendations = recommender.recommend(uid, n_recommendations=3)
    print('\nRecommandé :')
    for movie in recommendations:
        print(f'  ➡️  {movie["title"]} (prédit : {movie["predicted_rating"]})')

In [ ]:
# SAUVEGARDE DU MODÈLE
model_path = '../models/recommendation_model.pkl'
recommender.save(model_path)

# Vérification
import os
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f'📦 Taille du fichier : {size_mb:.2f} MB')

In [ ]:
# TEST DE CHARGEMENT
loaded = MovieRecommender.load(model_path)
test_result = loaded.recommend(user_id=1, n_recommendations=3)
print('Test de rechargement :')
for m in test_result:
    print(f'  ✅ {m["title"]}')